In [1]:
%pip install -q dotenv llama_stack_client==0.3.5 fire

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

from llama_stack_client import Agent
from llama_stack_client import AgentEventLogger
from llama_stack_client import LlamaStackClient

In [3]:
def stream_with_tools(response_stream):
    for chunk in response_stream:
        payload = getattr(getattr(chunk, "event", None), "payload", None)
        if not payload:
            continue
        step_type = getattr(payload, "step_type", "")
        if step_type == "tool_execution":
            print("\n🔧 Tool executed:")
            try:
                content = payload.delta.text
                print(content)
            except AttributeError:
                print(payload)
        elif getattr(payload, "event_type", "") == "step_progress":
            delta = getattr(getattr(payload, "delta", None), "text", None)
            if delta:
                print(delta, end="", flush=True)

In [4]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")
model = "vllm/qwen3-8b"
client = LlamaStackClient(
    base_url=base_url,
)

In [5]:
agent_config = {
    "model": "vllm/qwen3-8b",
    "instructions": """You are a helpful websearch assistant. When you are asked to search the latest you must use a tool. 
                     Whenever a tool is called, be sure return the response in a friendly and helpful tone.""",
    "enable_session_persistence": False,
    "toolgroups": ["mcp::customer", "mcp::finance"],
    "tool_choice": "auto",
    "sampling_params": {
        "max_tokens": 1024,
    },
}


agent_response = client.alpha.agents.create(agent_config=agent_config)
agent_id = agent_response.agent_id

session_response = client.alpha.agents.session.create(
    agent_id=agent_id,
    session_name="mcp_test_session"
)
session_id = session_response.session_id

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1alpha/agents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1alpha/agents/7a824f83-2796-4b2b-9784-a2196390b4a6/session "HTTP/1.1 200 OK"


In [6]:
question = "find more detail about franwilson@example.com"

response = client.alpha.agents.turn.create(
    agent_id=agent_id,
    session_id=session_id,
    messages=[{"role": "user", "content": question}],
    stream=True,
)

stream_with_tools(response)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1alpha/agents/7a824f83-2796-4b2b-9784-a2196390b4a6/session/df9d603e-c62a-4eff-8e44-f129dc6a9630/turn "HTTP/1.1 200 OK"


<think>
Okay, the user is asking for more details about the email address franwilson@example.com. Let me see which tools I can use here.

First, I check the available functions. There's search_customers, get_customer, fetch_order_history, and fetch_invoice_history. 

The search_customers function allows filtering by contact_email, so maybe I can use that. The get_customer function requires a customer_id, which I don't have yet. The other functions (order and invoice history) also need a customer_id. 

So the first step is to find the customer ID associated with that email. Using search_customers with contact_email set to franwilson@example.com might return customers matching that email. If there's a match, I can then use get_customer with the found customer ID to retrieve detailed information. 

Alternatively, maybe the search_customers function can return enough details directly. Let me check the description again. The search_customers function returns a list of customers matching the

In [7]:
question = "find all order and invoice about franwilson@example.com"

response = client.alpha.agents.turn.create(
    agent_id=agent_id,
    session_id=session_id,
    messages=[{"role": "user", "content": question}],
    stream=True,
)

stream_with_tools(response)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1alpha/agents/7a824f83-2796-4b2b-9784-a2196390b4a6/session/df9d603e-c62a-4eff-8e44-f129dc6a9630/turn "HTTP/1.1 200 OK"


<think>
Okay, the user wants all orders and invoices related to franwilson@example.com. Let me start by recalling the previous interaction. Earlier, they searched for the email and found Fran Wilson's customer details. Now, they need orders and invoices.

First, I need to get the customer ID from the previous search. The response showed the customer ID is LONEP. So, I'll use that ID to fetch order history and invoice history.

For the order history, the function fetch_order_history requires the customer_id. I'll call that function with customer_id "LONEP". Similarly, for invoices, fetch_invoice_history uses the same customer_id. I should make sure to call both functions with the correct parameters. Since the user wants all orders and invoices, I'll set the limit to a higher number, maybe 100, to ensure all records are retrieved. But wait, the default limit is 50. If there are more than 50, the user might need to adjust, but since they asked for all, I'll set a higher limit. However, th